# CT Log Sampler: Are Certificate Transparency Logs Useful for B2B Lead Gen?

A practical exploration. We pull a small sample from an old-style CT log, extract domain names, check if they're real websites, and try to classify whether they belong to a business.

**Author:** Isaac Bell
**Date:** 2026-08-30

---

### What this notebook does

1. Checks Chrome's list of known CT logs
2. Hits an old-style log (DigiCert Wyvern) to get its current size
3. Fetches a small batch of entries
4. Extracts domain names from certificate SANs
5. Checks DNS resolution, HTTP status, and classifies the homepage
6. Outputs a CSV with preliminary classification

**You can run this yourself** — no API keys required. Just a Python environment.

In [ ]:
# Setup
import json
import requests
import base64
import socket
import ssl
from urllib.parse import urlparse
from datetime import datetime
import csv

SAMPLE_SIZE = 100  # How many entries to fetch (start small)
LOG_URL = "https://wyvern.ct.digicert.com/2026h2"
STH_URL = f"{LOG_URL}/ct/v1/get-sth"
ENTRIES_URL = f"{LOG_URL}/ct/v1/get-entries"

## Step 1: Check the Log List

Google maintains a list of all recognized CT logs. Let's verify our target log is still usable.

In [ ]:
log_list_url = "https://www.gstatic.com/ct/log_list/v3/all_logs_list.json"
resp = requests.get(log_list_url)
log_data = resp.json()

# Find DigiCert's Wyvern log
for operator in log_data.get("operators", []):
    for log in operator.get("logs", []):
        if "wyvern" in log.get("url", "").lower():
            print(f"Found: {log.get('description', log['url'])}")
            print(f"  URL: {log['url']}")
            print(f"  State: {log.get('state', 'unknown')}")
            print(f"  Log ID: {log['log_id'][:40]}...")

print(f"\nTotal operators: {len(log_data.get('operators', []))}")
print(f"Total logs across all operators: {sum(len(op.get('logs', [])) for op in log_data.get('operators', []))}")

## Step 2: Get the Signed Tree Head

The STH tells us how many entries are in the log. We use this to decide where to sample from.

In [ ]:
resp = requests.get(STH_URL)
sth = resp.json()

tree_size = sth["tree_size"]
timestamp = sth["timestamp"]
human_time = datetime.fromtimestamp(timestamp / 1000).strftime("%Y-%m-%d %H:%M:%S")

print(f"Tree size: {tree_size:,} entries")
print(f"Timestamp: {human_time}")
print(f"\nThat's {tree_size / 1_000_000:.1f} million entries in this shard alone.")
print(f"Log covers roughly 6 months — started around {human_time[:10] if human_time else 'unknown'}.")

## Step 3: Fetch a Batch of Entries

We'll fetch from near the end of the log (recent entries) backwards. The API returns certificates in batches — we ask for `start` to `end`.

We're sampling from well into the log (after initial catch-up), not the very beginning.

In [ ]:
# Fetch from a recent position
# Start at tree_size - SAMPLE_SIZE to get the most recent entries
start = max(0, tree_size - SAMPLE_SIZE - 1)
end = tree_size - 1

print(f"Fetching entries {start:,} to {end:,} ({end - start + 1:,} entries)...")
resp = requests.get(f"{ENTRIES_URL}?start={start}&end={end}")
entries = resp.json()
print(f"Got {len(entries.get('entries', []))} entries")

## Step 4: Extract Domain Names

Each entry contains a certificate (or precertificate) which has Subject Alternative Names (SANs) — the domain names the certificate covers. This is where the lead data comes from.

Note: The certificate data is base64-encoded. We need to decode it and extract the SANs.

In [ ]:
import hashlib

def extract_leaf_cert(entry):
    """Extract the certificate (or precertificate) from a log entry."""
    leaf_input = entry.get("leaf_input", "")
    if not leaf_input:
        return None

    try:
        der_data = base64.b64decode(leaf_input)
        # The leaf_input is a MerkleTreeLeaf structure.
        # The certificate starts at a variable offset depending on the entry type.
        # For a full implementation, use a proper ASN.1 parser.
        # For this sampler, we'll look for the certificate data after the leaf type header.
        return der_data
    except Exception as e:
        print(f"  Failed to decode entry: {e}")
        return None

# Try to extract raw certs
raw_certs = []
for i, entry in enumerate(entries.get("entries", [])):
    leaf = extract_leaf_cert(entry)
    if leaf:
        raw_certs.append(leaf)

print(f"Extracted {len(raw_certs)} raw certificate entries out of {len(entries.get('entries', []))}")

## Step 5: Parse SANs with cryptography library

For proper parsing, we need to use the `cryptography` library to decode the X.509 certificate. Let's try parsing what we can.

In [ ]:
try:
    from cryptography import x509
    from cryptography.hazmat.backends import default_backend
    HAS_CRYPTO = True
except ImportError:
    HAS_CRYPTO = False
    print("cryptography library not installed. Install with: pip install cryptography")

# The entries returned by get-entries have the certificate in the 'leaf_input' field.
# The actual DER-encoded certificate is embedded inside the MerkleTreeLeaf structure.
# For a proper implementation, you'd need to parse the TLS-encoded MerkleTreeLeaf.
# For this sampler, we'll extract the domain names from the extra_data field if available.

# Let's look at the raw entry structure
if entries.get("entries"):
    first_entry = entries["entries"][0]
    print("Entry keys:", list(first_entry.keys()))
    print(f"leaf_input length: {len(first_entry.get('leaf_input', ''))} chars")
    if 'extra_data' in first_entry:
        print(f"extra_data length: {len(first_entry.get('extra_data', ''))} chars")

## Deep Dive: The Actual Certificate Data Structure

The raw data from a CT log entry is TLS-encoded, not JSON. The `leaf_input` contains a MerkleTreeLeaf structure which wraps the certificate. To properly extract domains, we need a CT client library.

For the sampler, we have two options:
1. Use a proper CT library (like `ct-python` or `certvalidator`)
2. Use certstream's WebSocket API instead, which gives us pre-parsed domains

Let's try option 2 — certstream gives us parsed data directly:

In [ ]:
# Try certstream for parsed domain data
import websocket
import json as json_lib
import threading
from datetime import datetime

CERTSTREAM_URL = "wss://certstream.calidog.io/"

collected_certs = []
MAX_CERTS = 20  # Limit for this sample
TIMEOUT_SECONDS = 30

print(f"Connecting to certstream to collect {MAX_CERTS} certificates...")
print("This may take a moment. Certstream sends live certificates as they're issued.")
print("(If this hangs, certstream may be down. Fall back to manual entry processing.)")

def on_message(ws, message):
    data = json_lib.loads(message)
    if data.get("message_type") == "certificate_update":
        leaf = data.get("data", {}).get("leaf_cert", {})
        all_domains = leaf.get("all_domains", [])
        if all_domains:
            entry = {
                "domains": all_domains,
                "not_before": leaf.get("not_before"),
                "not_after": leaf.get("not_after"),
                "issuer": leaf.get("issuer", {}).get("O", "unknown"),
                "seen": datetime.now().isoformat()
            }
            collected_certs.append(entry)
            print(f"  [{len(collected_certs)}/{MAX_CERTS}] {all_domains[0]} ({len(all_domains)} domains) — {leaf.get('issuer', {}).get('O', '?')}")

    if len(collected_certs) >= MAX_CERTS:
        ws.close()

def on_error(ws, error):
    print(f"WebSocket error: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f"Connection closed")

try:
    ws = websocket.WebSocketApp(
        CERTSTREAM_URL,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )
    
    # Run in a thread with timeout
    thread = threading.Thread(target=ws.run_forever, kwargs={'ping_interval': 10, 'ping_timeout': 5})
    thread.daemon = True
    thread.start()
    
    thread.join(timeout=TIMEOUT_SECONDS)
except ImportError:
    print("websocket-client not installed. Install with: pip install websocket-client")
except Exception as e:
    print(f"Certstream error: {e}")

print(f"\nCollected {len(collected_certs)} certificates with {sum(len(c['domains']) for c in collected_certs)} total domains")
print(f"\nSample domains:")
for c in collected_certs[:5]:
    print(f"  {c['domains'][0]}")

## Step 6: DNS Resolution Check

For each domain, check if it resolves to an IP address. This is the cheapest filter — domains that don't resolve aren't real websites.

In [ ]:
def domain_resolves(domain):
    """Check if a domain has at least one A or AAAA record."""
    try:
        socket.getaddrinfo(domain, 80, socket.AF_INET, socket.SOCK_STREAM)
        return True
    except (socket.gaierror, socket.timeout):
        return False

results = []
for c in collected_certs:
    domain = c["domains"][0]  # Use the primary domain (first in list)
    resolves = domain_resolves(domain)
    results.append({
        "domain": domain,
        "resolves": resolves,
        "all_domains": c["domains"],
        "issuer": c["issuer"]
    })

resolving = [r for r in results if r["resolves"]]
not_resolving = [r for r in results if not r["resolves"]]

print(f"Domains that resolve: {len(resolving)}/{len(results)} ({len(resolving)/len(results)*100:.0f}%)")
print(f"Domains that DON'T resolve: {len(not_resolving)}/{len(results)} ({len(not_resolving)/len(results)*100:.0f}%)")
print()

print("Non-resolving domains (first 10):")
for r in not_resolving[:10]:
    print(f"  {r['domain']}")

print()
print("Resolving domains (first 10):")
for r in resolving[:10]:
    print(f"  {r['domain']}")

## Step 7: HTTP Check

For domains that resolved, try to fetch the homepage and check the response.

In [ ]:
def check_http(domain, timeout=5):
    """Fetch the homepage and return status code and a preview of the response."""
    for scheme in ['https://', 'http://']:
        try:
            url = scheme + domain
            resp = requests.get(url, timeout=timeout, allow_redirects=True, headers={
                "User-Agent": "Mozilla/5.0 (compatible; LeadsDB-Sampler/1.0)"
            })
            return {
                "status": resp.status_code,
                "final_url": resp.url,
                "content_type": resp.headers.get("Content-Type", ""),
                "title": extract_title(resp.text),
                "content_preview": resp.text[:500] if resp.status_code < 400 else ""
            }
        except requests.exceptions.RequestException:
            continue
    return {"status": 0, "final_url": "", "content_type": "", "title": "", "content_preview": ""}

def extract_title(html):
    """Extract the <title> tag from HTML."""
    import re
    match = re.search(r'<title[^>]*>(.*?)</title>', html, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else ""

# Check HTTP for resolving domains
for r in results:
    if r["resolves"]:
        http_result = check_http(r["domain"])
        r["http_status"] = http_result["status"]
        r["title"] = http_result["title"]
        r["content_preview"] = http_result["content_preview"]
        r["content_type"] = http_result["content_type"]
    else:
        r["http_status"] = 0
        r["title"] = ""
        r["content_preview"] = ""
        r["content_type"] = ""

# Show results
print(f"{'Domain':<40} {'DNS':<6} {'HTTP':<6} {'Title'}")
print("-" * 90)
for r in results[:20]:
    dns = "✓" if r["resolves"] else "✗"
    http = str(r["http_status"]) if r["http_status"] else "—"
    title = r["title"][:50] if r["title"] else ""
    print(f"{r['domain']:<40} {dns:<6} {http:<6} {title}")

## Step 8: Preliminary Classification

Based on the HTTP response, we can make a rough guess about whether this is a real business.

Indicators of a real business:
- Returns 200 OK
- Title mentions a real company name
- Content has pricing, team, services, or product copy
- Has a privacy policy, about page, terms of service

Indicators of noise:
- DNS doesn't resolve
- Returns a generic hosting/godaddy landing page
- Title says "Domain is for sale" or similar
- Returns 4xx or 5xx
- AWS/Cloudflare default pages

In [ ]:
def classify_domain(r):
    """Simple heuristic classification. Returns a tuple (signal_type, confidence)."""
    if not r["resolves"]:
        return ("dead", 0.9)
    
    http_status = r.get("http_status", 0)
    title = r.get("title", "").lower()
    content = r.get("content_preview", "").lower()
    domain = r.get("domain", "").lower()
    
    # Parked / for sale
    parked_signals = ["for sale", "domain is parked", "buy this domain", "this domain may be for sale",
                      "godaddy", "sedo", "hugedomains"]
    for s in parked_signals:
        if s in title or s in content:
            return ("parked", 0.8)
    
    # Generic hosting placeholder
    hosting_signals = ["default page", "hosting", "plesk", "cpanel", "apache2", "nginx",
                      "site not configured", "website is under construction", "coming soon",
                      "this is a placeholder"]
    for s in hosting_signals:
        if s in title or s in content[:200]:
            return ("hosting_placeholder", 0.7)
    
    # Infrastructure / not a real site
    infra_signals = ["amazonaws.com", "cloudfront.net", "azurewebsites.net", "herokuapp.com",
                    "firebaseapp.com", "netlify.app", "vercel.app", "pages.dev",
                    "amplifyapp.com", "elasticbeanstalk.com"]
    for s in infra_signals:
        if s in domain:
            return ("infrastructure", 0.8)
    
    # Likely real business indicators
    business_signals = ["about us", "contact us", "pricing", "services", "products", "our team",
                       "privacy policy", "terms of service", "sign in", "login", "get started",
                       "book now", "schedule", "portfolio", "case studies"]
    business_score = sum(1 for s in business_signals if s in content)
    if business_score >= 2:
        return ("likely_business", 0.6 + 0.1 * min(business_score, 4))
    
    if http_status == 200 and title:
        return ("unknown_site", 0.4)  # Could be anything
    
    return ("unknown", 0.3)

# Classify all domains
classifications = {}
for r in results:
    cls, confidence = classify_domain(r)
    r["classification"] = cls
    r["confidence"] = confidence
    classifications[cls] = classifications.get(cls, 0) + 1

print("Classification results:")
for cls, count in sorted(classifications.items(), key=lambda x: -x[1]):
    print(f"  {cls}: {count} ({count/len(results)*100:.0f}%)")

print()
print("Potential leads (likely_business):")
leads = [r for r in results if r["classification"] == "likely_business"]
for l in leads:
    print(f"  {l['domain']} — {l['title'][:60]}")

## Step 9: Export Results

Save the results as a CSV so you can inspect, share, or load into a spreadsheet.

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"ct_sampler_results_{timestamp}.csv"

with open(filename, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["domain", "resolves", "http_status", "title", "classification", "confidence", "issuer", "all_domains"])
    writer.writeheader()
    for r in results:
        writer.writerow({
            "domain": r["domain"],
            "resolves": r["resolves"],
            "http_status": r.get("http_status", ""),
            "title": r.get("title", ""),
            "classification": r["classification"],
            "confidence": r["confidence"],
            "issuer": r["issuer"],
            "all_domains": ", ".join(r["all_domains"])
        })

print(f"Saved {len(results)} results to {filename}")
print(f"\nSummary:")
print(f"  Total domains sampled: {len(results)}")
print(f"  DNS-resolving: {len(resolving)}")
print(f"  HTTP 200: {sum(1 for r in results if r.get('http_status') == 200)}")
print(f"  Likely businesses: {len(leads)}")
print(f"  Dead/parked/unknown: {len(results) - len(leads)}")
print(f"\nSignal-to-noise ratio: {len(leads)}/{len(results)} ({len(leads)/len(results)*100:.0f}% if all leads are real)")

## Interpretation

This is a tiny sample. A few dozen certificates from one log at one point in time. But it gives us a starting point.

**Key numbers to watch:**
- What percentage of certificates resolve to real websites?
- What percentage of those belong to actual businesses?
- What does the noise look like? (Dead domains, parked pages, hosting defaults — can we filter them cheaply?)
- Are there patterns in the false positives we can learn from?

**Next step after this sampler:** Increase the sample size to 1,000-5,000 entries. Add LLM-based classification (send homepage preview to a cheap model). If the signal-to-noise ratio is above 5-10%, the full pipeline makes sense.

---

*Notebook generated 2026-08-30. Part of the [LeadsDB V2](https://github.com/IsaacBell/leads-db) project.*